# TotalSegmentator × FiftyOne — Curate & Debug 3D Medical Segmentation

**A self-contained demo.** This notebook loads the official
[TotalSegmentator](https://github.com/wasserth/totalsegmentator) data, turns 3D CT/MR
volumes into something FiftyOne can visualize, runs the TotalSegmentator model to
produce predictions, evaluates them against ground truth **per anatomical structure**,
and then uses FiftyOne to surface the *worst* cases — the actual point of a data-curation
tool.

The story arc:

1. **Get data** — the official 102-subject CT subset (small, purpose-built for exploration).
2. **Slice** each 3D volume into 2D frames so it renders as a scrollable "video" in the App,
   carrying the ground-truth organ masks.
3. **Load** into a FiftyOne dataset with rich per-subject metadata.
4. **Predict** — run the real TotalSegmentator model on each volume (MPS/CUDA/CPU).
5. **Evaluate** — per-structure IoU / Dice via `evaluate_segmentations()`.
6. **Curate** — sort to the failure cases, find the structures the model struggles with,
   and build a review view. This is the money shot.

> **Before you run anything:** this notebook needs its own dedicated virtual environment
> and its own Jupyter kernel. Follow **Section 0 · Environment setup** immediately below —
> it takes two minutes and prevents the dependency clashes (torch, nnU-Net, FiftyOne) that
> otherwise plague medical-imaging setups.
>
> **This demo uses real data and real model predictions.** It uses the real CT volumes
> and runs the actual TotalSegmentator model, so the evaluation numbers are a genuine
> measurement. There is no synthetic fallback — the notebook requires the real dataset.
> Every expensive stage (extract, slice, inference, evaluation) is **cached and skipped**
> if its output already exists, so re-running is cheap.


## 0 · Environment setup (dedicated venv + Jupyter kernel)

Run these steps **once** in a terminal *before* executing any cells. They create an
isolated environment and register it as a selectable Jupyter kernel named
**`ts-fiftyone`**.

### macOS / Linux

```bash
# 1. Create the environment (Python 3.10 or 3.11 recommended; avoid 3.14)
python3.11 -m venv .venv-ts-fiftyone
source .venv-ts-fiftyone/bin/activate

# 2. Core libraries (pin FiftyOne to the 1.19 line)
pip install --upgrade pip
pip install "fiftyone==1.19.*" nibabel numpy pillow requests ipykernel jupyterlab

# 3. The model. On Apple silicon the default torch wheel already has MPS support.
pip install TotalSegmentator torch

# 4. Register THIS venv as a Jupyter kernel called "ts-fiftyone"
python -m ipykernel install --user \
    --name ts-fiftyone \
    --display-name "Python (ts-fiftyone)"
```

### Windows (PowerShell)

```powershell
py -3.11 -m venv .venv-ts-fiftyone
.venv-ts-fiftyone\Scripts\Activate.ps1
pip install --upgrade pip
pip install "fiftyone==1.19.*" nibabel numpy pillow requests ipykernel jupyterlab
pip install TotalSegmentator torch
python -m ipykernel install --user --name ts-fiftyone --display-name "Python (ts-fiftyone)"
```

That last command is the important one: it writes a kernelspec whose interpreter is
*this* venv's `python`, so anything you `pip install` here is what the notebook sees.

### Then, in Jupyter, select the kernel

- **JupyterLab / Notebook 7:** open this file, then use the kernel picker at the
  **top-right** of the notebook (it may say *Python 3 (ipykernel)*) → choose
  **`Python (ts-fiftyone)`**.
- **VS Code:** click the kernel selector at the top-right → *Select Another Kernel…* →
  *Jupyter Kernel…* → **`Python (ts-fiftyone)`**. (If it doesn't appear, run
  *Developer: Reload Window*.)
- **Classic Notebook:** menu **Kernel → Change kernel → Python (ts-fiftyone)**.

If you don't see it, confirm it registered:

```bash
jupyter kernelspec list        # should list  ts-fiftyone
```

To remove it later: `jupyter kernelspec uninstall ts-fiftyone`.

> **Launch Jupyter from the activated venv too**, so the server and kernel agree:
> ```bash
> source .venv-ts-fiftyone/bin/activate     # Windows: .venv-ts-fiftyone\Scripts\Activate.ps1
> jupyter lab
> ```

The next cell **verifies** you're on the right kernel before doing any work.


In [ ]:

# --- Kernel / environment sanity check -------------------------------------
# Confirms this notebook is running inside a dedicated venv with the right packages,
# so you fail fast with a clear message instead of hitting a confusing ImportError.
import sys, os, importlib.util
from pathlib import Path

print("Python executable:", sys.executable)
print("Python version:   ", sys.version.split()[0])

# 1) Are we in *a* virtual environment (not the system Python)?
in_venv = (
    sys.prefix != getattr(sys, "base_prefix", sys.prefix)
    or "VIRTUAL_ENV" in os.environ
)
if not in_venv:
    print("\n[!] You do NOT appear to be in a virtual environment.")
    print("    Select the 'Python (ts-fiftyone)' kernel (see Section 0), then re-run.")
else:
    venv_root = os.environ.get("VIRTUAL_ENV", sys.prefix)
    print("Virtual env:      ", venv_root)

# 2) Is FiftyOne importable and on the 1.19 line?
spec = importlib.util.find_spec("fiftyone")
if spec is None:
    print("\n[!] `fiftyone` is not installed in this kernel.")
    print("    You're likely on the wrong kernel. Pick 'Python (ts-fiftyone)' and re-run,")
    print("    or complete Section 0's install steps in the activated venv.")
else:
    import fiftyone as _fo
    ok = _fo.__version__.startswith("1.19")
    print("FiftyOne version: ", _fo.__version__, "(OK)" if ok else "(expected 1.19.x)")

assert in_venv and spec is not None, (
    "Wrong environment/kernel. Follow Section 0 and select the 'Python (ts-fiftyone)' "
    "kernel before running the rest of the notebook."
)
print("\nEnvironment looks good. Proceed.")

## 0b · Imports (and a safety-net installer)

If you completed Section 0, everything is already installed in this kernel and the cell
below just imports. The `ensure()` helper is a safety net: it installs into **this
kernel's** interpreter only if something is missing, so it can't pollute other
environments.


In [ ]:

import sys, subprocess, importlib

def ensure(pkg, import_name=None, extra=None):
    """Install a package into the *current* interpreter if missing."""
    name = import_name or pkg
    try:
        return importlib.import_module(name)
    except ImportError:
        cmd = [sys.executable, "-m", "pip", "install", "-q", pkg]
        if extra:
            cmd += extra
        subprocess.check_call(cmd)
        return importlib.import_module(name)

# Hard requirements
fo_mod   = ensure("fiftyone")
nib      = ensure("nibabel")
np_mod   = ensure("numpy", "numpy")
pil_mod  = ensure("pillow", "PIL")
req_mod  = ensure("requests")

import fiftyone as fo
import fiftyone.brain as fob          # optional, used later if available
from fiftyone import ViewField as F
import numpy as np
import nibabel as nib
from PIL import Image

print("fiftyone", fo.__version__)

## 1 · Configuration

`MAX_SUBJECTS` keeps the live demo snappy. The full 102-subject subset works too, it just
takes longer to slice. `AXIS` picks the slicing plane (2 = axial, the conventional
radiology view). `SLICE_STRIDE` subsamples slices along the volume so each "video" is a
manageable length.


In [ ]:

import os
from pathlib import Path

WORKDIR      = Path("./ts_fiftyone_demo").resolve()
DATA_DIR     = WORKDIR / "data"
FRAMES_DIR   = WORKDIR / "frames"     # 2D slice PNGs (images shown in the App)
GT_MASK_DIR  = WORKDIR / "masks_gt"   # ground-truth label PNGs (single-channel)
PR_MASK_DIR  = WORKDIR / "masks_pred" # predicted label PNGs
for d in (DATA_DIR, FRAMES_DIR, GT_MASK_DIR, PR_MASK_DIR):
    d.mkdir(parents=True, exist_ok=True)

MAX_SUBJECTS = 6      # bump toward 102 for a fuller dataset
AXIS         = 2      # 0=sagittal, 1=coronal, 2=axial
SLICE_STRIDE = 3      # keep every Nth slice that contains a labeled structure
DATASET_NAME = "totalsegmentator-ct-demo"

# --- Data source -------------------------------------------------------------
# By default the notebook auto-downloads the official small CT subset (102 subjects,
# ~3 GB) from Zenodo — no manual step required. If you've already downloaded the zip,
# set the TS_LOCAL_ZIP environment variable to its path and it'll be used instead of
# downloading again.
SUBSET_ZENODO = "https://zenodo.org/records/8367169"

_env_zip = os.environ.get("TS_LOCAL_ZIP", "").strip()
LOCAL_ZIP = Path(_env_zip).expanduser() if _env_zip else None   # None => just download

print("Workdir:", WORKDIR)
if LOCAL_ZIP:
    print("Local zip:", LOCAL_ZIP, "| exists:", LOCAL_ZIP.exists())
else:
    print("No local zip set (TS_LOCAL_ZIP) — will download the subset if not already extracted.")

## 2 · Get the data (no re-download, no re-extract)

TotalSegmentator volumes are one folder per subject:

```
s0011/
  ct.nii.gz                      # the CT volume
  segmentations/
    liver.nii.gz                 # one binary mask per structure
    spleen.nii.gz
    ...
```

Resolution order, cheapest first:

1. **Already extracted** under `DATA_DIR` → use as-is, do nothing.
2. **Local zip** → only if you set the `TS_LOCAL_ZIP` environment variable to a
   pre-downloaded copy; extracted once into `DATA_DIR`.
3. **Download** from Zenodo (the default) → the 102-subject subset, ~3 GB, one time.

There is no synthetic fallback: this demo requires the real data. The download is public
(CC-BY-4.0) and fully automatic, so a first run on a fresh laptop just works.


In [ ]:

import zipfile, csv, urllib.parse, re

def _subjects_present(root: Path):
    """List subject dirs (those containing ct.nii.gz) anywhere under root."""
    return sorted({p.parent for p in root.rglob("ct.nii.gz")})

def _flatten_into(dest: Path):
    """If subjects live one level deep (dest/<wrapper>/s0011/ct.nii.gz), lift them
    up to dest/s0011 so all downstream globbing is uniform. Idempotent."""
    direct = [p.parent for p in dest.glob("*/ct.nii.gz")]
    if direct:
        return  # already at the expected depth
    for wrapper in [d for d in dest.iterdir() if d.is_dir()]:
        if any(wrapper.glob("*/ct.nii.gz")):
            for s in [p.parent for p in wrapper.glob("*/ct.nii.gz")]:
                target = dest / s.name
                if not target.exists():
                    s.rename(target)

def _extract_zip(zip_path: Path, dest: Path):
    print(f"Extracting {zip_path.name} ({zip_path.stat().st_size/1e9:.2f} GB) -> {dest}")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(dest)
    _flatten_into(dest)

def _zenodo_zip_url(record_url):
    html = req_mod.get(record_url, timeout=30).text
    m = re.search(r'(/records/\d+/files/[^"\']+\.zip)\?download=1', html)
    if not m:
        raise RuntimeError("Could not locate a .zip on the Zenodo page")
    return urllib.parse.urljoin(record_url, m.group(1) + "?download=1")

def prepare_data(dest: Path) -> list:
    # 1) already extracted?
    subs = _subjects_present(dest)
    if subs:
        print(f"Data already extracted: {len(subs)} subjects. Nothing to do.")
        return subs
    # 2) local zip? (only if TS_LOCAL_ZIP was set)
    if LOCAL_ZIP is not None and LOCAL_ZIP.exists():
        _extract_zip(LOCAL_ZIP, dest)
        subs = _subjects_present(dest)
        if subs:
            print(f"Extracted local zip: {len(subs)} subjects.")
            return subs
        raise RuntimeError(f"{LOCAL_ZIP} extracted but no ct.nii.gz found inside.")
    # 3) download (default path)
    print("Downloading the 102-subject CT subset from Zenodo (~3 GB, one time) ...")
    url = _zenodo_zip_url(SUBSET_ZENODO)
    tmp_zip = dest / "_subset.zip"
    if not tmp_zip.exists():
        blob = req_mod.get(url, timeout=1800).content
        tmp_zip.write_bytes(blob)
    _extract_zip(tmp_zip, dest)
    subs = _subjects_present(dest)
    if not subs:
        raise RuntimeError("Download+extract produced no subjects.")
    return subs

all_subjects = prepare_data(DATA_DIR)
subjects = all_subjects[:MAX_SUBJECTS]
print(f"Using {len(subjects)} of {len(all_subjects)} subject(s):", [s.name for s in subjects])

In [ ]:

# Load per-subject metadata from meta.csv if the dataset shipped one.
meta_by_subject = {}
meta_csv = next(DATA_DIR.rglob("meta.csv"), None)
if meta_csv:
    with open(meta_csv, newline="") as fh:
        sample = fh.read(2048); fh.seek(0)
        delim = ";" if sample.count(";") >= sample.count(",") else ","
        for row in csv.DictReader(fh, delimiter=delim):
            key = row.get("image_id") or row.get("subject") or row.get("id")
            if key:
                meta_by_subject[key.strip()] = row
    cols = list(next(iter(meta_by_subject.values())).keys()) if meta_by_subject else []
    print(f"Loaded metadata for {len(meta_by_subject)} subjects; columns: {cols}")
else:
    print("No meta.csv found in dataset.")

## 3 · Choose the structures to track

TotalSegmentator's `total` task has 117 classes; evaluating and rendering all of them at
once is noisy. We pick a focused, clinically meaningful panel of abdominal organs. Each
gets a distinct integer id — that integer becomes the pixel value in the label masks and
the key in FiftyOne's `mask_targets`.


In [ ]:

# structure name -> integer label id (0 is reserved for background)
STRUCTURES = {
    "liver":         1,
    "spleen":        2,
    "kidney_left":   3,
    "kidney_right":  4,
    "pancreas":      5,
    "stomach":       6,
    "gallbladder":   7,
    "aorta":         8,
}
MASK_TARGETS = {0: "background", **{v: k for k, v in STRUCTURES.items()}}

# A stable, readable color per structure for the App.
PALETTE = {
    "background": "#000000", "liver": "#d1495b", "spleen": "#edae49",
    "kidney_left": "#00798c", "kidney_right": "#2e4057", "pancreas": "#66a182",
    "stomach": "#8d5a97", "gallbladder": "#c8b8db", "aorta": "#e3170a",
}
print(MASK_TARGETS)

## 4 · Volume → 2D frames (the key trick)

FiftyOne doesn't render raw NIfTI volumes directly; the standard pattern
([per Voxel51's own guidance](https://docs.voxel51.com/getting_started/medical_imaging/index.html))
is to **slice a volume into 2D frames** and represent it as a *video* sample. Each frame is:

- a windowed grayscale PNG of the CT slice (the visible media), plus
- a single-channel PNG label mask where each pixel holds the structure's integer id.

We window CT to a soft-tissue range (about −150…250 HU) so organs are visible, skip empty
slices, and combine all per-structure binary masks into one multi-class mask per slice.


In [ ]:

def window_ct(slice2d, lo=-150.0, hi=250.0):
    """Map HU window to 0..255 grayscale."""
    x = np.clip(slice2d.astype(np.float32), lo, hi)
    x = (x - lo) / (hi - lo) * 255.0
    return x.astype(np.uint8)

def load_gt_multiclass(subject: Path, shape):
    """Combine available per-structure binary NIfTIs into one int label volume."""
    lbl = np.zeros(shape, dtype=np.uint8)
    segdir = subject / "segmentations"
    for name, idx in STRUCTURES.items():
        f = segdir / f"{name}.nii.gz"
        if f.exists():
            m = np.asanyarray(nib.load(str(f)).dataobj) > 0
            if m.shape == shape:
                lbl[m] = idx
    return lbl

def take_slice(vol, axis, i):
    return np.take(vol, i, axis=axis)

def orient(a):
    """Rotate slice for a conventional upright view."""
    return np.rot90(a)

In [ ]:

import fiftyone as fo

def build_samples_from_existing(subject):
    """Reconstruct samples from already-written PNGs (no NIfTI load)."""
    fdir = FRAMES_DIR / subject.name
    out = []
    for img_p in sorted(fdir.glob("slice_*.png")):
        i = int(img_p.stem.split("_")[1])
        msk_p = GT_MASK_DIR / subject.name / img_p.name
        if not msk_p.exists():
            continue
        s = fo.Sample(filepath=str(img_p))
        s["subject"] = subject.name
        s["slice_index"] = i
        s["ground_truth"] = fo.Segmentation(mask_path=str(msk_p))
        meta = meta_by_subject.get(subject.name, {})
        for col in ("age", "sex", "institution", "manufacturer", "split", "study_type"):
            if col in meta and meta[col] not in ("", None):
                s[col] = meta[col]
        out.append(s)
    return out

def slice_subject(subject):
    """Slice a volume into per-slice PNGs (only the labeled slices) and return samples."""
    ct = np.asanyarray(nib.load(str(subject / "ct.nii.gz")).dataobj)
    gt = load_gt_multiclass(subject, ct.shape)
    (FRAMES_DIR / subject.name).mkdir(parents=True, exist_ok=True)
    (GT_MASK_DIR / subject.name).mkdir(parents=True, exist_ok=True)
    n = ct.shape[AXIS]
    out = []
    for i in range(0, n, SLICE_STRIDE):
        gslice = orient(take_slice(gt, AXIS, i))
        if not gslice.any():
            continue  # skip slices with no labeled structure
        gray = orient(window_ct(take_slice(ct, AXIS, i)))
        img_p = FRAMES_DIR / subject.name / f"slice_{i:04d}.png"
        msk_p = GT_MASK_DIR / subject.name / f"slice_{i:04d}.png"
        if not img_p.exists():
            Image.fromarray(gray).save(img_p)
        if not msk_p.exists():
            Image.fromarray(gslice.astype(np.uint8)).save(msk_p)
        s = fo.Sample(filepath=str(img_p))
        s["subject"] = subject.name
        s["slice_index"] = int(i)
        s["ground_truth"] = fo.Segmentation(mask_path=str(msk_p))
        meta = meta_by_subject.get(subject.name, {})
        for col in ("age", "sex", "institution", "manufacturer", "split", "study_type"):
            if col in meta and meta[col] not in ("", None):
                s[col] = meta[col]
        out.append(s)
    return out

# One image sample per labeled slice, tagged with `subject` so you can group/scroll
# the stack in the App. Renders instantly in the grid. Idempotent: subjects already
# sliced are rebuilt from PNGs on disk without re-reading the (large) NIfTI volumes.
samples = []
for subject in subjects:
    fdir = FRAMES_DIR / subject.name
    if fdir.exists() and any(fdir.glob("slice_*.png")):
        subj_samples = build_samples_from_existing(subject)
        print(f"{subject.name}: reused {len(subj_samples)} slices from disk")
    else:
        subj_samples = slice_subject(subject)
        print(f"{subject.name}: sliced {len(subj_samples)} labeled slices")
    samples.extend(subj_samples)

print("Total slice-samples:", len(samples))

## 5 · Build the FiftyOne dataset

We attach `mask_targets` and a color scheme so the App shows structure *names* on hover
and consistent colors. Every slice knows its `subject`, so you can use **dynamic groups**
in the App to scroll a subject's slices like a stack.


In [ ]:

# Set REBUILD = True to force a clean rebuild of the FiftyOne dataset.
REBUILD = False

if fo.dataset_exists(DATASET_NAME) and not REBUILD:
    dataset = fo.load_dataset(DATASET_NAME)
    if len(dataset) == len(samples):
        print(f"Loaded existing dataset '{DATASET_NAME}' with {len(dataset)} samples "
              f"(unchanged; skipping rebuild).")
    else:
        print(f"Existing dataset has {len(dataset)} samples but {len(samples)} expected "
              f"-> rebuilding.")
        fo.delete_dataset(DATASET_NAME)
        dataset = None
else:
    if fo.dataset_exists(DATASET_NAME):
        fo.delete_dataset(DATASET_NAME)
    dataset = None

if dataset is None:
    dataset = fo.Dataset(DATASET_NAME, persistent=True)
    dataset.add_samples(samples)

# Names on hover + evaluation semantics (cheap; always set)
dataset.mask_targets = {"ground_truth": MASK_TARGETS, "prediction": MASK_TARGETS}
dataset.default_mask_targets = MASK_TARGETS
dataset.app_config.color_scheme = fo.ColorScheme(
    color_by="value",
    fields=[{
        "path": "ground_truth",
        "maskTargetsColors": [
            {"intTarget": idx, "color": PALETTE[name]}
            for idx, name in MASK_TARGETS.items()
        ],
    }],
)
dataset.save()
print(dataset)

## 6 · Real predictions from TotalSegmentator

**This is the real model.** We run TotalSegmentator on each CT volume via its Python API
and produce genuine predicted organ masks — the numbers in the evaluation section are then
a real measurement of model quality, not a simulation.

Design choices that make this robust:

- **Per-structure output, not `--ml`.** We let TotalSegmentator write one binary NIfTI per
  organ (`liver.nii.gz`, `spleen.nii.gz`, …) — the *same* filenames the ground truth uses.
  We then reuse the exact `load_gt_multiclass()` helper to assemble the prediction volume,
  so predicted and ground-truth label ids are guaranteed to line up. No fragile remapping.
- **`roi_subset`** restricts inference to our 8 structures, which greatly cuts runtime.
- **`--fast`** runs the 3 mm model; drop it for full 1.5 mm accuracy.
- **Device:** on Apple-silicon Macs we use `mps` (the maintainers note it gives a big
  speedup); otherwise CUDA if present, else CPU.

Weights (~hundreds of MB for the `total` task) download automatically on first run.


### 6a · Install the model and pick a device

Installs `TotalSegmentator` into this kernel if it isn't already importable. On CPU this
is slow (~1 min/volume even with `--fast`); on an M-series Mac `mps` is much faster.


In [ ]:

# Install the real model + torch if needed. On Apple silicon, the default pip torch
# wheel already includes MPS support.
totalseg = ensure("TotalSegmentator", "totalsegmentator")
torch    = ensure("torch")

from totalsegmentator.python_api import totalsegmentator

def pick_device():
    try:
        if torch.cuda.is_available():
            return "gpu"
        if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
            return "mps"
    except Exception:
        pass
    return "cpu"

DEVICE = pick_device()
FAST   = True   # 3mm model; set False for full-resolution 1.5mm predictions
print("TotalSegmentator ready. device =", DEVICE, "| fast =", FAST)

### 6b · Run inference and assemble prediction masks

For each subject we run the model once (writing per-organ NIfTIs), then reuse
`load_gt_multiclass()` to build a prediction volume whose label ids match the ground
truth exactly. We then write the same per-slice PNG masks we used for the GT.


In [ ]:

import traceback

ROI = list(STRUCTURES.keys())

# Reuse the SAME assembler as the ground truth so label ids are guaranteed consistent.
# TotalSegmentator writes <name>.nii.gz directly into raw_out; the shim makes
# load_gt_multiclass look there instead of a "segmentations" subfolder.
class _Shim:
    def __init__(self, d): self._d = d
    def __truediv__(self, x):
        return self._d if x == "segmentations" else (self._d / x)

def inference_done(raw_out: Path) -> bool:
    """True if TotalSegmentator output for our ROI already exists (any organ file)."""
    return raw_out.is_dir() and any((raw_out / f"{name}.nii.gz").exists() for name in ROI)

pred_volumes = {}   # subject.name -> predicted multiclass volume (our label ids)
failures = []

for subject in subjects:
    raw_out = PR_MASK_DIR / (subject.name + "_ts")   # per-organ NIfTIs land here
    try:
        if inference_done(raw_out):
            print(f"{subject.name}: predictions already on disk -> skipping model run")
        else:
            raw_out.mkdir(parents=True, exist_ok=True)
            print(f"Running TotalSegmentator on {subject.name} ...")
            totalsegmentator(
                str(subject / "ct.nii.gz"),
                str(raw_out),
                fast=FAST,
                roi_subset=ROI,
                device=DEVICE,
                quiet=True,
            )
        ref_shape = np.asanyarray(nib.load(str(subject / "ct.nii.gz")).dataobj).shape
        pred_volumes[subject.name] = load_gt_multiclass(_Shim(raw_out), ref_shape)
    except Exception as e:
        failures.append((subject.name, repr(e)))
        traceback.print_exc()

print(f"\nInference ready for {len(pred_volumes)}/{len(subjects)} subjects.")
if failures:
    for name, err in failures:
        print("  FAILED", name, "->", err)

In [ ]:

# HARD GATE: this notebook is about *real* predictions. If inference produced nothing,
# stop here rather than proceeding with empty predictions.
assert pred_volumes, (
    "No real predictions were produced. Check the install/device in 6a and re-run 6b. "
    "Ensure the real CT data extracted correctly and TotalSegmentator is importable."
)

# Write per-slice prediction PNGs aligned to the kept slices, and attach to samples.
# Idempotent: existing prediction PNGs are reused; samples already carrying a
# prediction with a valid mask_path are left untouched.
written = 0
for s in dataset.iter_samples(autosave=True, progress=True):
    subj = s["subject"]; i = s["slice_index"]
    if subj not in pred_volumes:
        continue
    pr_dir = PR_MASK_DIR / subj
    pr_dir.mkdir(parents=True, exist_ok=True)
    pr_path = pr_dir / f"slice_{i:04d}.png"
    if not pr_path.exists():
        pslice = orient(take_slice(pred_volumes[subj], AXIS, i))
        Image.fromarray(pslice.astype(np.uint8)).save(pr_path)
        written += 1
    # Always (re)assign: same mask_path each run, so this is idempotent. We avoid
    # s.get_field("prediction"), which RAISES AttributeError until the field exists
    # on the dataset schema (i.e. on the very first sample of a fresh run).
    s["prediction"] = fo.Segmentation(mask_path=str(pr_path))

n_pred = len(dataset.exists("prediction"))
print(f"Prediction PNGs written this run: {written}. "
      f"Real predictions attached to {n_pred}/{len(dataset)} slices.")

## 7 · Evaluate — per-structure, not just an average

`evaluate_segmentations()` does pixelwise multi-class evaluation and, with `mask_targets`,
reports **per-class IoU / precision / recall**. It also writes per-sample accuracy fields
we can sort on. This is where aggregate numbers become actionable: you immediately see
which organs the model handles and which it drops.


In [ ]:

EVAL_KEY = "eval"

if EVAL_KEY in dataset.list_evaluations():
    print(f"Evaluation '{EVAL_KEY}' already exists -> loading cached results.")
    results = dataset.load_evaluation_results(EVAL_KEY)
else:
    results = dataset.evaluate_segmentations(
        "prediction",
        gt_field="ground_truth",
        eval_key=EVAL_KEY,
        mask_targets=MASK_TARGETS,
    )

# Per-class table (background excluded from the interesting rows)
results.print_report()

In [ ]:

# Rank structures by IoU so the weakest ones jump out.
report = results.report()          # dict: class -> {precision, recall, f1-score, support}
rows = [(cls, d) for cls, d in report.items()
        if cls not in ("background", "micro avg", "macro avg", "weighted avg")]
rows.sort(key=lambda kv: kv[1].get("f1-score", 0.0))

print(f"{'structure':<14}{'precision':>10}{'recall':>10}{'f1':>8}{'support':>10}")
for cls, d in rows:
    print(f"{cls:<14}{d['precision']:>10.3f}{d['recall']:>10.3f}"
          f"{d['f1-score']:>8.3f}{int(d['support']):>10}")

## 8 · Curate — find and review the failures

The payoff. `evaluate_segmentations()` stamped each sample with an `eval_accuracy`
(and precision/recall) field. We:

1. **Sort worst-first** to build a triage queue.
2. **Filter** to slices where a specific hard structure (say, `pancreas`) is present but
   the model did poorly — a targeted review set.
3. Optionally compute **uniqueness/embeddings** to spot redundant or outlier slices worth
   annotating.

Everything below is a `DatasetView` you can open directly in the App.


In [ ]:

# 1) Worst slices overall — the triage queue
worst = dataset.sort_by("eval_accuracy")           # ascending: worst first
print("Lowest-accuracy slices:")
for s in worst[:5]:
    print(f"  {s['subject']}  slice {s['slice_index']:>4}  acc={s['eval_accuracy']:.3f}")

# 2) A targeted review set: slices that CONTAIN a hard structure but scored poorly.
#    We flag presence by checking the GT mask for that class id at load-time would be ideal;
#    here we use per-sample accuracy plus a subject filter as a stand-in that always works.
HARD = "pancreas"
hard_id = STRUCTURES[HARD]

def gt_contains(sample, cls_id):
    return bool((np.array(Image.open(sample["ground_truth"].mask_path)) == cls_id).any())

ids = [s.id for s in dataset if gt_contains(s, hard_id)]
review = (dataset.select(ids)
                 .sort_by("eval_accuracy")
                 .limit(25))
print(f"\nReview set: {len(review)} slices containing '{HARD}', worst first.")

In [ ]:

# 3) (Optional) uniqueness to prioritize diverse slices for annotation review.
try:
    if "uniqueness" not in dataset.list_brain_runs():
        fob.compute_uniqueness(dataset)          # writes a `uniqueness` field
        print("Computed `uniqueness`.")
    else:
        print("`uniqueness` already computed -> skipping.")
    diverse_review = review.sort_by("uniqueness", reverse=True)
    print("Review set re-sorted by diversity.")
except Exception as e:
    diverse_review = review
    print("Skipped uniqueness (optional):", repr(e))

## 8b · Save all the demos as named views

So you're not hand-sorting the App every time you present, we persist the key views with
`save_view()`. They then appear in the App's **view selector** (top-left, where it says
"Unsaved view") — one click each.

First we stamp a lightweight `gt_structures` field on every sample: the list of structure
names present in that slice's ground truth. On-disk segmentation masks can't be filtered
by pixel value directly in the App, so this per-sample list is what makes
"show me every slice containing the pancreas" a fast, clickable filter.


In [ ]:

from fiftyone import ViewField as F

def stamp_structures(ds, struct_map, field="gt_structures"):
    """Add a list field naming the structures present in each sample's GT mask.
    Idempotent + cached: skips if already populated."""
    id_to_name = {v: k for k, v in struct_map.items()}
    already = field in ds.get_field_schema()
    if already and ds.count(F(field).length() > 0) == len(ds):
        print(f"`{field}` already populated -> skipping.")
        return
    for s in ds.iter_samples(autosave=True, progress=True):
        m = np.array(Image.open(s["ground_truth"].mask_path))
        present = [id_to_name[v] for v in np.unique(m) if v in id_to_name]
        s[field] = present
    ds.save()
    print(f"Stamped `{field}` on {len(ds)} samples.")

stamp_structures(dataset, STRUCTURES)

In [ ]:

def build_saved_views(ds, struct_map, eval_key, hard=("pancreas", "gallbladder"),
                      uniqueness=True):
    """Create a standard suite of named saved views on `ds`. Overwrites if present."""
    acc = f"{eval_key}_accuracy"
    rec = f"{eval_key}_recall"
    prc = f"{eval_key}_precision"

    def put(name, view, desc):
        if ds.has_saved_view(name):
            ds.delete_saved_view(name)
        try:
            ds.save_view(name, view, description=desc)
        except TypeError:
            # older/newer signature without description kwarg
            ds.save_view(name, view)

    # Triage queues
    put("worst_overall", ds.sort_by(acc),
        "All slices, lowest pixel accuracy first — the model's worst predictions.")
    put("best_overall", ds.sort_by(acc, reverse=True),
        "Highest accuracy first — clean big-organ cases for contrast.")
    put("under_segmented", ds.sort_by(rec),
        "Lowest recall first — where the model MISSES structures (drops/erodes).")
    put("over_segmented", ds.sort_by(prc),
        "Lowest precision first — where the model OVER-calls structures.")

    # Per-hard-structure review sets (present in GT, worst first)
    for name in hard:
        if name in struct_map:
            v = ds.match(F("gt_structures").contains(name)).sort_by(acc)
            put(f"review_{name}", v,
                f"Slices containing '{name}', worst accuracy first — targeted review.")

    # Diversity-aware failure set
    if uniqueness and "uniqueness" in ds.list_brain_runs():
        primary = hard[0] if hard and hard[0] in struct_map else None
        base = (ds.match(F("gt_structures").contains(primary)) if primary else ds)
        v = base.sort_by(acc).limit(50).sort_by("uniqueness", reverse=True)
        put("diverse_failures",
            v, "Worst ~50 (optionally for a hard structure), re-ranked by diversity "
               "so you review a spread, not near-duplicates.")

    print("Saved views:", ds.list_saved_views())

build_saved_views(dataset, STRUCTURES, eval_key="eval")

## 9 · Launch the App

Open the dataset and pick a view from the **view selector** (top-left). Suggested demo arc:

1. **`worst_overall`** — the ten-second money shot. Open the top slice in the modal and
   toggle `prediction` vs `ground_truth` to see the model erode or drop an organ.
2. **`best_overall`** — flip to the clean liver/aorta cases; the competence gap is obvious.
3. **`review_pancreas`** / **`review_gallbladder`** — targeted failure queues for the hard,
   small structures. Some "errors" here are actually ground-truth label noise — a great
   thing to catch live.
4. **`under_segmented`** vs **`over_segmented`** — different failure modes (misses vs
   over-calls) via recall- and precision-sorted views.
5. **`diverse_failures`** — a spread of failures for building a smart re-annotation set,
   not 30 near-identical bad slices from one patient.

Also try: **Histograms** panel on `eval_accuracy`, `sex`, `institution` to hunt for
scanner/demographic bias.


In [ ]:

session = fo.launch_app(dataset)
session.view = dataset.load_saved_view("worst_overall")   # start on the triage queue
print("App launched. Views available:", dataset.list_saved_views())
print("Switch views from the top-left selector, or in Python:")
print("  session.view = dataset.load_saved_view('review_pancreas')")

In [ ]:

# Keep the process alive when running as a script (no-op in Jupyter).
# session.wait()

# Bonus Demo A · Evaluate *every* structure in the data

The main demo tracked 8 organs. But the 102-subject zip already ships ground-truth masks
for **all** TotalSegmentator structures — we just didn't use them. Here we auto-discover
every structure actually present in the data, run the model over all of them, and produce
a much richer per-class failure report. **No new download.**

Why this is the highest-impact upgrade: the per-structure IoU gradient gets far more
interesting when it spans tiny/variable targets (adrenal glands, gallbladder, small
vessels) alongside easy ones (liver, aorta). That spread is the story radiology-AI teams
actually care about.

> Runtime note: dropping `roi_subset` means the model segments everything, which is slower
> per volume. We keep `MAX_SUBJECTS` small and cache aggressively. On CPU this is the
> slow part — MPS strongly recommended.


In [ ]:

# Discover the union of structures present across our subjects' ground truth.
# This is version-robust: whatever the data actually contains is what we evaluate.
def discover_structures(subjects):
    names = set()
    for subj in subjects:
        segdir = subj / "segmentations"
        if segdir.is_dir():
            for f in segdir.glob("*.nii.gz"):
                names.add(f.name[:-len(".nii.gz")])
    return sorted(names)

available = discover_structures(subjects)
print(f"{len(available)} structures present in the ground truth, e.g.:")
print(", ".join(available[:20]), "...")

# Assign integer ids 1..N (0 = background). Cap for a snappy demo; raise as you like.
# Kept <= 255 so masks fit in a standard 8-bit PNG (what FiftyOne reads most reliably).
MAX_STRUCTS_ALL = 60
ALL_STRUCTURES = {name: i + 1 for i, name in enumerate(available[:MAX_STRUCTS_ALL])}
ALL_TARGETS = {0: "background", **{v: k for k, v in ALL_STRUCTURES.items()}}
print(f"\nEvaluating {len(ALL_STRUCTURES)} structures (id 1..{len(ALL_STRUCTURES)}).")

In [ ]:

# Generalized helpers parameterized by a {name: id} structure map.
# (The main demo's load_gt_multiclass hard-coded STRUCTURES; these take it as an arg.)
def build_multiclass(segdir_owner, shape, struct_map):
    """segdir_owner supports `/ 'segmentations' / '<name>.nii.gz'` (real subject or _Shim)."""
    lbl = np.zeros(shape, dtype=np.uint8)    # <=255 structures (see MAX_STRUCTS_ALL cap)
    for name, idx in struct_map.items():
        f = (segdir_owner / "segmentations") / f"{name}.nii.gz"
        if Path(str(f)).exists():
            m = np.asanyarray(nib.load(str(f)).dataobj) > 0
            if m.shape == shape:
                lbl[m] = idx
    return lbl

def slice_volume_multiclass(subject, struct_map, frames_dir, gt_dir, tag):
    """Write per-slice grayscale + multiclass GT PNGs for one subject; return samples."""
    ct = np.asanyarray(nib.load(str(subject / "ct.nii.gz")).dataobj)
    gt = build_multiclass(subject, ct.shape, struct_map)
    (frames_dir / subject.name).mkdir(parents=True, exist_ok=True)
    (gt_dir / subject.name).mkdir(parents=True, exist_ok=True)
    out = []
    for i in range(0, ct.shape[AXIS], SLICE_STRIDE):
        gslice = orient(take_slice(gt, AXIS, i))
        if not gslice.any():
            continue
        img_p = frames_dir / subject.name / f"{tag}_{i:04d}.png"
        msk_p = gt_dir / subject.name / f"{tag}_{i:04d}.png"
        if not img_p.exists():
            Image.fromarray(orient(window_ct(take_slice(ct, AXIS, i)))).save(img_p)
        if not msk_p.exists():
            # PNG supports 16-bit single-channel; FiftyOne reads these as int masks
            Image.fromarray(gslice.astype(np.uint8)).save(msk_p)
        s = fo.Sample(filepath=str(img_p))
        s["subject"] = subject.name
        s["slice_index"] = int(i)
        s["ground_truth"] = fo.Segmentation(mask_path=str(msk_p))
        out.append(s)
    return out

In [ ]:

# Directories + dataset name for the all-structures demo
ALL_FRAMES = WORKDIR / "frames_all"
ALL_GTDIR  = WORKDIR / "masks_gt_all"
ALL_PRDIR  = WORKDIR / "masks_pred_all"
for d in (ALL_FRAMES, ALL_GTDIR, ALL_PRDIR):
    d.mkdir(parents=True, exist_ok=True)
ALL_DATASET = "totalsegmentator-ct-allstructs"

# 1) Slice (idempotent)
all_samples = []
for subject in subjects:
    fdir = ALL_FRAMES / subject.name
    if fdir.exists() and any(fdir.glob("all_*.png")):
        # rebuild from disk
        reused = 0
        for img_p in sorted(fdir.glob("all_*.png")):
            i = int(img_p.stem.split("_")[1])
            msk_p = ALL_GTDIR / subject.name / img_p.name
            if not msk_p.exists():
                continue
            s = fo.Sample(filepath=str(img_p))
            s["subject"] = subject.name; s["slice_index"] = i
            s["ground_truth"] = fo.Segmentation(mask_path=str(msk_p))
            all_samples.append(s); reused += 1
        print(f"{subject.name}: reused {reused} slices")
    else:
        subj_samples = slice_volume_multiclass(subject, ALL_STRUCTURES, ALL_FRAMES, ALL_GTDIR, "all")
        all_samples.extend(subj_samples)
        print(f"{subject.name}: sliced {len(subj_samples)} slices (all structures)")

print("Total all-structure slice-samples:", len(all_samples))

In [ ]:

# 2) Inference over ALL structures (no roi_subset). Cached per subject.
all_pred_volumes = {}
for subject in subjects:
    raw_out = ALL_PRDIR / (subject.name + "_ts_all")
    try:
        have = raw_out.is_dir() and any((raw_out / f"{n}.nii.gz").exists()
                                        for n in list(ALL_STRUCTURES)[:5])
        if have:
            print(f"{subject.name}: all-structure predictions cached -> skip")
        else:
            raw_out.mkdir(parents=True, exist_ok=True)
            print(f"Running TotalSegmentator (ALL structures) on {subject.name} ...")
            totalsegmentator(str(subject / "ct.nii.gz"), str(raw_out),
                             fast=FAST, device=DEVICE, quiet=True)   # no roi_subset
        ref_shape = np.asanyarray(nib.load(str(subject / "ct.nii.gz")).dataobj).shape
        all_pred_volumes[subject.name] = build_multiclass(_Shim(raw_out), ref_shape, ALL_STRUCTURES)
    except Exception as e:
        print("  FAILED", subject.name, "->", repr(e))

assert all_pred_volumes, "No all-structure predictions produced."
print(f"Predictions ready for {len(all_pred_volumes)}/{len(subjects)} subjects.")

In [ ]:

# 3) Build dataset + attach predictions
REBUILD_ALL = False
if fo.dataset_exists(ALL_DATASET) and not REBUILD_ALL:
    ds_all = fo.load_dataset(ALL_DATASET)
    if len(ds_all) != len(all_samples):
        fo.delete_dataset(ALL_DATASET); ds_all = None
else:
    if fo.dataset_exists(ALL_DATASET):
        fo.delete_dataset(ALL_DATASET)
    ds_all = None
if ds_all is None:
    ds_all = fo.Dataset(ALL_DATASET, persistent=True)
    ds_all.add_samples(all_samples)

ds_all.mask_targets = {"ground_truth": ALL_TARGETS, "prediction": ALL_TARGETS}
ds_all.default_mask_targets = ALL_TARGETS
ds_all.save()

for s in ds_all.iter_samples(autosave=True, progress=True):
    subj, i = s["subject"], s["slice_index"]
    if subj not in all_pred_volumes:
        continue
    (ALL_PRDIR / subj).mkdir(parents=True, exist_ok=True)
    pr_path = ALL_PRDIR / subj / f"all_{i:04d}.png"
    if not pr_path.exists():
        pslice = orient(take_slice(all_pred_volumes[subj], AXIS, i))
        Image.fromarray(pslice.astype(np.uint8)).save(pr_path)
    s["prediction"] = fo.Segmentation(mask_path=str(pr_path))

print(f"All-structures dataset ready: {len(ds_all)} slices, "
      f"{len(ALL_STRUCTURES)} classes.")

In [ ]:

# 4) Per-structure evaluation across all classes
ALL_EVAL = "eval_all"
if ALL_EVAL in ds_all.list_evaluations():
    res_all = ds_all.load_evaluation_results(ALL_EVAL)
else:
    res_all = ds_all.evaluate_segmentations(
        "prediction", gt_field="ground_truth",
        eval_key=ALL_EVAL, mask_targets=ALL_TARGETS,
    )

# Rank every structure by F1 so the model's best/worst targets are explicit.
rep = res_all.report()
rows = [(c, d) for c, d in rep.items()
        if c not in ("background", "micro avg", "macro avg", "weighted avg")]
rows.sort(key=lambda kv: kv[1].get("f1-score", 0.0))

print(f"{'structure':<28}{'precision':>10}{'recall':>9}{'f1':>8}{'support':>10}")
print("-" * 65)
for c, d in rows:
    if d.get("support", 0) == 0:
        continue
    print(f"{c:<28}{d['precision']:>10.3f}{d['recall']:>9.3f}"
          f"{d['f1-score']:>8.3f}{int(d['support']):>10}")
print("\nWorst 5:", [c for c, _ in rows if _.get('support',0)>0][:5])
print("Best 5: ", [c for c, _ in rows if _.get('support',0)>0][-5:])

In [ ]:

# 5) Save views for the all-structures demo, then open it.
# Reuse the helpers from Section 8b (stamp_structures, build_saved_views).
stamp_structures(ds_all, ALL_STRUCTURES)

# Pick the two worst-scoring structures (with support) as targeted review sets — this is
# more compelling than hard-coding, since it adapts to what the model actually struggles
# with on your data.
worst_structs = tuple(c for c, d in rows if d.get("support", 0) > 0)[:2]
print("Auto-selected worst structures for review views:", worst_structs)

build_saved_views(ds_all, ALL_STRUCTURES, eval_key="eval_all",
                  hard=worst_structs, uniqueness=True)

session_all = fo.launch_app(ds_all)
session_all.view = ds_all.load_saved_view("worst_overall")
print("All-structures App launched on `worst_overall`.")
print("Views:", ds_all.list_saved_views())

# Bonus Demo B · 3D context via orthogonal planes (grouped dataset)

The main demo shows one plane (axial). Radiologists read in **three** planes at once.
FiftyOne's *grouped datasets* let us store axial + coronal + sagittal views of the same
volume as a single group, so you can flip between planes in the App and see the same
structures from three perspectives — genuine 3D context without a mesh viewer.

We reuse the volumes already on disk. For each subject we take the mid-volume slice along
each of the three axes, window it, and render the ground-truth mask. Each subject becomes
one group with slices `axial`, `coronal`, `sagittal`.

> This is the fastest robust path to "3D" in the App. For full volumetric rendering you'd
> export an orthographic-projection scene or a mesh, but orthogonal planes give you the
> multi-view intuition immediately and always render.


In [ ]:

ORTHO_FRAMES = WORKDIR / "frames_ortho"
ORTHO_GT     = WORKDIR / "masks_ortho"
for d in (ORTHO_FRAMES, ORTHO_GT):
    d.mkdir(parents=True, exist_ok=True)
ORTHO_DATASET = "totalsegmentator-ct-3planes"

# Reuse the focused 8-organ map from the main demo for clean, colorful masks.
PLANES = {"axial": 2, "coronal": 1, "sagittal": 0}

def mid_labeled_index(gt, axis):
    """Pick the slice along `axis` with the most labeled pixels (most informative)."""
    counts = [np.count_nonzero(np.take(gt, k, axis=axis)) for k in range(gt.shape[axis])]
    return int(np.argmax(counts)) if max(counts) > 0 else gt.shape[axis] // 2

def render_plane(ct, gt, axis, out_img, out_msk):
    # Note: a single rot90 gives a reasonable upright view for axial; coronal/sagittal
    # may appear rotated/flipped vs. radiological convention. That's a display nicety,
    # not a correctness issue — masks and image are sliced identically so they align.
    idx = mid_labeled_index(gt, axis)
    gray = orient(window_ct(np.take(ct, idx, axis=axis)))
    mask = orient(np.take(gt, idx, axis=axis)).astype(np.uint8)
    if not out_img.exists():
        Image.fromarray(gray).save(out_img)
    if not out_msk.exists():
        Image.fromarray(mask).save(out_msk)
    return idx

In [ ]:

if fo.dataset_exists(ORTHO_DATASET):
    fo.delete_dataset(ORTHO_DATASET)
ds3 = fo.Dataset(ORTHO_DATASET, persistent=True)
ds3.add_group_field("group", default="axial")

group_samples = []
for subject in subjects:
    ct = np.asanyarray(nib.load(str(subject / "ct.nii.gz")).dataobj)
    gt = build_multiclass(subject, ct.shape, STRUCTURES)   # 8-organ map from main demo
    (ORTHO_FRAMES / subject.name).mkdir(parents=True, exist_ok=True)
    (ORTHO_GT / subject.name).mkdir(parents=True, exist_ok=True)
    grp = fo.Group()
    for plane, axis in PLANES.items():
        img_p = ORTHO_FRAMES / subject.name / f"{plane}.png"
        msk_p = ORTHO_GT / subject.name / f"{plane}.png"
        idx = render_plane(ct, gt, axis, img_p, msk_p)
        s = fo.Sample(filepath=str(img_p), group=grp.element(plane))
        s["subject"] = subject.name
        s["plane"] = plane
        s["slice_index"] = idx
        s["ground_truth"] = fo.Segmentation(mask_path=str(msk_p))
        group_samples.append(s)
    print(f"{subject.name}: 3 orthogonal planes")

ds3.add_samples(group_samples)
ds3.mask_targets = {"ground_truth": MASK_TARGETS}
ds3.default_mask_targets = MASK_TARGETS
ds3.app_config.color_scheme = fo.ColorScheme(
    color_by="value",
    fields=[{"path": "ground_truth",
             "maskTargetsColors": [{"intTarget": idx, "color": PALETTE[name]}
                                   for idx, name in MASK_TARGETS.items()]}],
)
ds3.save()
print(ds3)
print("Group slices:", ds3.group_slices)

In [ ]:

# Open the grouped dataset. In the App, the group modal shows all three planes together;
# use the slice selector (axial / coronal / sagittal) to flip perspectives, and toggle
# `ground_truth` to see the same organs across planes.
session_3d = fo.launch_app(ds3)
print("3-plane grouped App launched. Open a group to see axial+coronal+sagittal together.")

# Bonus Demo C · Catching vertebra mix-ups (the `vertebrae_pp` story)

TotalSegmentator's author recently shipped a `vertebrae_pp` task that fixes a real pain
point: individual vertebrae sometimes got **mislabeled** — part of L3 tagged as L2, two
adjacent vertebrae swapped, or one vertebra split across two labels. The fix is
connected-component renumbering so each vertebra gets one consistent, correctly-ordered
label.

This is a perfect FiftyOne demo because a vertebra mix-up is a **label-identity error, not
a shape error**. Standard Dice/IoU can look *fine* — the pixels are roughly in the right
place, they're just wrongly *named*. But if you color each vertebra by an
anatomically-ordered label id (C1 at the top → sacrum at the bottom), a correct spine is a
smooth top-to-bottom color **gradient**, and any mix-up is an obvious **break** in that
gradient. The eye catches it instantly where a metric hides it.

**What we build:** for each subject, one **sagittal** slice (the whole spine in a single
image), with two mask fields on the same sample:

- `vert_correct` — vertebrae assembled in canonical C1→sacrum order (the "after":
  what `vertebrae_pp` gives you)
- `vert_mixed` — the same spine with a simulated classic mix-up (adjacent-label swaps
  + one split), illustrating the "before"

> Honesty note: the "before" here is a *simulated* illustration of the mix-up failure
> mode, so the demo runs on the data you already have without needing an old model
> version. To show real model output, run `totalsegmentator(..., task="vertebrae_pp")`
> for the "after" and an older version (or the plain per-vertebra assembly) for the
> "before"; everything downstream is identical. No new download either way.


In [ ]:

# Canonical head-to-toe vertebra order -> label id increases inferiorly.
# A correct spine is therefore a smooth color gradient; a mix-up breaks it.
VERT_ORDER = (
    [f"vertebrae_C{i}" for i in range(1, 8)] +      # C1..C7
    [f"vertebrae_T{i}" for i in range(1, 13)] +     # T1..T12
    [f"vertebrae_L{i}" for i in range(1, 6)] +      # L1..L5
    ["vertebrae_S1", "sacrum"]                      # sacrum region (either name)
)
VERT_MAP = {name: i + 1 for i, name in enumerate(VERT_ORDER)}
VERT_TARGETS = {0: "background", **{v: k for k, v in VERT_MAP.items()}}

# A perceptually-ordered colormap: hue sweeps with position down the spine, so correct
# ordering reads as a rainbow and swaps/splits jump out.
import colorsys
def _hue_hex(t):
    r, g, b = colorsys.hsv_to_rgb(0.85 * t, 0.85, 0.95)  # 0..1 down the spine
    return "#%02x%02x%02x" % (int(r*255), int(g*255), int(b*255))
VERT_PALETTE = {"background": "#000000"}
_names = [n for n in VERT_ORDER]
for k, name in enumerate(_names):
    VERT_PALETTE[name] = _hue_hex(k / max(1, len(_names) - 1))

print(f"{len(VERT_MAP)} vertebra labels, C1(1) -> sacrum({len(VERT_MAP)}).")

In [ ]:

def make_mixed(vol, seed=0):
    """Simulate the classic mix-up: swap a couple of adjacent vertebra labels and split
    one vertebra into two labels. Operates on the multiclass volume; geometry unchanged,
    only identities scrambled — exactly what vertebrae_pp fixes."""
    rng = np.random.default_rng(seed)
    out = vol.copy()
    present = sorted(int(v) for v in np.unique(vol) if v != 0)
    if len(present) < 4:
        return out
    # 1) swap two randomly-chosen adjacent labels
    for _ in range(2):
        i = int(rng.integers(0, len(present) - 1))
        a, b = present[i], present[i + 1]
        ma, mb = vol == a, vol == b
        out[ma] = b
        out[mb] = a
    # 2) split one vertebra: relabel its lower half as its inferior neighbour's id
    j = int(rng.integers(1, len(present) - 1))
    vid = present[j]
    mask = vol == vid
    if mask.any():
        # find extent along the inferior-superior axis (AXIS=2 is axial => use axis 2 index)
        zs = np.where(mask.any(axis=(0, 1)))[0]
        if len(zs) > 3:
            mid = zs[len(zs) // 2]
            lower = mask.copy()
            lower[:, :, :mid] = False
            out[lower] = present[min(j + 1, len(present) - 1)]
    return out

In [ ]:

# Directories + dataset
VERT_FRAMES = WORKDIR / "frames_vert"
VERT_CORRECT = WORKDIR / "masks_vert_correct"
VERT_MIXED   = WORKDIR / "masks_vert_mixed"
for d in (VERT_FRAMES, VERT_CORRECT, VERT_MIXED):
    d.mkdir(parents=True, exist_ok=True)
VERT_DATASET = "totalsegmentator-vertebrae-mixup"

SAG_AXIS = 0   # sagittal plane: whole spine visible as one vertical column

if fo.dataset_exists(VERT_DATASET):
    fo.delete_dataset(VERT_DATASET)
dsv = fo.Dataset(VERT_DATASET, persistent=True)

vert_samples = []
for subject in subjects:
    ct = np.asanyarray(nib.load(str(subject / "ct.nii.gz")).dataobj)
    vol = build_multiclass(subject, ct.shape, VERT_MAP)     # correct, ordered labels
    if not vol.any():
        print(f"{subject.name}: no vertebra labels present -> skipped")
        continue
    mixed = make_mixed(vol, seed=abs(hash(subject.name)) % (2**32))

    # pick the sagittal slice with the most vertebra pixels (spine most visible)
    counts = [np.count_nonzero(np.take(vol, k, axis=SAG_AXIS)) for k in range(vol.shape[SAG_AXIS])]
    idx = int(np.argmax(counts))

    (VERT_FRAMES / subject.name).mkdir(parents=True, exist_ok=True)
    img_p = VERT_FRAMES / subject.name / "sagittal.png"
    cor_p = VERT_CORRECT / f"{subject.name}.png"
    mix_p = VERT_MIXED / f"{subject.name}.png"
    if not img_p.exists():
        Image.fromarray(orient(window_ct(np.take(ct, idx, axis=SAG_AXIS),
                                         lo=-200, hi=1000))).save(img_p)  # bone window
    if not cor_p.exists():
        Image.fromarray(orient(np.take(vol, idx, axis=SAG_AXIS)).astype(np.uint8)).save(cor_p)
    if not mix_p.exists():
        Image.fromarray(orient(np.take(mixed, idx, axis=SAG_AXIS)).astype(np.uint8)).save(mix_p)

    s = fo.Sample(filepath=str(img_p))
    s["subject"] = subject.name
    s["vert_correct"] = fo.Segmentation(mask_path=str(cor_p))  # "after" (vertebrae_pp)
    s["vert_mixed"]   = fo.Segmentation(mask_path=str(mix_p))  # "before" (mix-up)
    vert_samples.append(s)
    print(f"{subject.name}: sagittal spine rendered")

dsv.add_samples(vert_samples)
dsv.mask_targets = {"vert_correct": VERT_TARGETS, "vert_mixed": VERT_TARGETS}
dsv.default_mask_targets = VERT_TARGETS
dsv.app_config.color_scheme = fo.ColorScheme(
    color_by="value",
    fields=[{"path": f, "maskTargetsColors":
             [{"intTarget": idx, "color": VERT_PALETTE[name]}
              for idx, name in VERT_TARGETS.items()]}
            for f in ("vert_correct", "vert_mixed")],
)
dsv.save()
print(dsv)

In [ ]:

# Quantify the mix-up honestly: a per-vertebra "identity confusion" count.
# For each correct label, how many pixels the MIXED map assigned to a DIFFERENT label.
# Off-diagonal mass = mix-ups. This is the number that sits next to the visual.
def identity_confusion(correct_dir, mixed_dir, subjects):
    swapped = 0
    total = 0
    for subj in subjects:
        cp = correct_dir / f"{subj.name}.png"
        mp = mixed_dir / f"{subj.name}.png"
        if not (cp.exists() and mp.exists()):
            continue
        c = np.array(Image.open(cp)); m = np.array(Image.open(mp))
        fg = c != 0
        total += int(fg.sum())
        swapped += int(((c != m) & fg).sum())
    return swapped, total

sw, tot = identity_confusion(VERT_CORRECT, VERT_MIXED, subjects)
if tot:
    print(f"Vertebra pixels with WRONG identity in the 'before' map: "
          f"{sw:,} / {tot:,} ({100*sw/tot:.1f}%)")
    print("In the 'after' (vert_correct) map this is 0% by construction — that's the fix.")

In [ ]:

# Save before/after views and launch. In the App, toggle vert_mixed vs vert_correct on
# the sagittal image: the mixed map shows a broken color sequence (swaps/splits), the
# correct map is a smooth C1->sacrum gradient.
for name in ("spine_before_mixed", "spine_after_correct"):
    if dsv.has_saved_view(name):
        dsv.delete_saved_view(name)
dsv.save_view("spine_before_mixed", dsv.select_fields("vert_mixed"))
dsv.save_view("spine_after_correct", dsv.select_fields("vert_correct"))

session_vert = fo.launch_app(dsv)
print("Vertebra mix-up App launched.")
print("Toggle `vert_mixed` (before) vs `vert_correct` (after) on the sagittal view.")
print("Correct = smooth top-to-bottom color gradient; mixed = broken sequence.")

# Where to take it next (needs additional downloads)

The two bonus demos above use **only the data you already have**. These further
extensions require fetching another dataset:

- **Full CT set (1228 subjects)** — download the full
  [CT dataset](https://zenodo.org/records/10047292) (~23 GB, CC-BY-4.0), set
  `TS_LOCAL_ZIP` to the downloaded file (or extract it into the data dir)
  at it, and raise `MAX_SUBJECTS`. Everything else is unchanged.
- **MRI (616 images)** — the [MR dataset](https://zenodo.org/records/14710732) (~5 GB,
  CC-BY-NC-SA, non-commercial). Two code changes beyond the download: run the model with
  `task="total_mr"`, and replace the CT-specific `window_ct()` (tuned to −150…250 HU) with
  an intensity normalization suited to MR (e.g. percentile clipping), since MR has no HU
  scale. The eval/curation half is identical.

> Not a medical device; for research/demo use only.
